# RSM338 Assignment 6 Part A: Classification
## March 25, 2026
### Ethan Wang, Kevin Yang

## 1. Data Preparation (Reused from Assignment 5)

In [1]:
# Load in the data
import pandas as pd

df = pd.read_excel('lending_clubFull_Data_Set.xlsx')
df

,Unnamed: 0,id,member_id,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,...,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,0,263591,545710,20000.0,60 months,17.93,342.94,E,E5,Wylie ISD,...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN
1,1,1613916,69664096,30000.0,36 months,11.99,996.29,C,C1,Sergeant,...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN
2,2,818934,8965180,21500.0,36 months,11.99,714.01,B,B3,Designer,...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN
3,3,1606612,70572960,10000.0,36 months,13.67,340.18,C,C3,NaN,...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN
4,4,1639932,68589517,5000.0,36 months,8.49,157.82,B,B1,Sr. Manufacturing Engineer,...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,24995,901888,4974773,10500.0,36 months,10.16,339.60,B,B1,Schneider Electric,...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN
24996,24996,945413,1279809,12000.0,36 months,14.33,412.06,C,C1,Clark County School District,...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN
24997,24997,366944,63496281,25000.0,36 months,12.69,838.63,C,C2,sales consultant,...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN
24998,24998,1043529,98124387,12200.0,60 months,13.49,280.66,C,C2,NaN,...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,NaN


Now that we loaded the data, we must  choose useful features for our analysis. As stated, many of the columns are redundant, so we should try to make our dataset more managable before our analysis.

### 1.1 Defining the Target Variable
We need a binary outcome for the loans - default or not - but we are given several outcomes. Let's first see all the unique loan outcomes.

In [2]:
# Display all unique values in the 'loan_status' column
unique_loan_status = df['loan_status'].unique()
unique_loan_status

array(['Charged Off', 'Current', 'Fully Paid', 'Late (31-120 days)',
       'In Grace Period', 'Late (16-30 days)',
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off', 'Default',
       nan], dtype=object)

After some brief research, we decided to define the following loan outcomes as default:

**['Charged Off', 'Late (31-120 days)', 'Does not meet the credit policy. Status:Charged Off']**

We believe that these are either explicitly defaulted, or highly likely to be defaulted. The rest of the outcomes are either paid off, being paid on time, or still recoverable. Going forward in our code, default loans will be represented using a dummy equal to 1.

### 1.2 Selecting Features
The features we have decided to select are:

[*'delinq_2yrs', 'annual_inc', 'num_accts_ever_120_pd', 'pct_tl_nvr_dlq', 'dti', 'loan_amnt', 'home_ownership', 'emp_length'*]

These features can be grouped as follows:

**Past Behaviour**

*'delinq_2yrs' (delinquencies in the last 2 years)* and *'pct_tl_nvr_dlq' (percent of trades never delinquent)* tell us about the borrower's previous behaviour. Having one or more delinquencies in the past, especially relatively recent ones, signal higher credit risk. It tells us that the borrower may have questionable self-judgement on their ability to repay debts.

**Repaying Power**

*'annual_inc' (annual income)* and *'dti'(monthly debt-to-income ratio)* tells us about the earning power of the borrower. Even if a borrower has unfortunate circumstances that may affect their payments, if they have a strong basis for their ability to repay, then it is very possible for the debt to be recovered over time.

**Stability**

*'home_ownership' (ownership status i.e., rent, mortgage)* and *'emp_length' (in years, 0-10)* tell us if the borrower has a stable life. Borrowers with stability in their lives will have much more predictable repayments compared to one who is constantly moving and/or job-hopping.

**Loan Risk**

*'loan_amnt' (the amount applied for by borrower)* tells us about the borrower's perceived debt capacity. If a borrower has external signals that suggest credit risk, but they perceive the repayments to be feasible, it may suggest the borrower has an appetite for risk, which is unattractive to lendors. It also tells us the overall burden of the loan on the borrower, not just if they can make payments in the short-term.

### 1.3 Handling Missing Values and Data Types 

In [3]:
# 1. Handle missing values (Example: dropping rows with any NAs in selected features)
features_to_use = ['delinq_2yrs', 'annual_inc', 'num_accts_ever_120_pd', 'pct_tl_nvr_dlq', 'dti', 'loan_amnt', 'home_ownership', 'emp_length', 'loan_status']
df_cleaned = df[features_to_use].dropna()
df_cleaned.head()

,delinq_2yrs,annual_inc,num_accts_ever_120_pd,pct_tl_nvr_dlq,dti,loan_amnt,home_ownership,emp_length,loan_status
1,3.0,136000.0,1.0,90.0,20.63,30000.0,MORTGAGE,10+ years,Current
2,0.0,50000.0,0.0,100.0,29.62,21500.0,RENT,1 year,Fully Paid
4,0.0,88000.0,1.0,60.0,5.32,5000.0,MORTGAGE,10+ years,Current
5,0.0,38500.0,0.0,100.0,33.73,16150.0,RENT,10+ years,Charged Off
6,0.0,40000.0,0.0,100.0,19.11,18400.0,RENT,8 years,Current


Theoretically, no applicant should have missing values in any of the columns. Missing values here, like income, would be highly problematic, as it should just be 0, not NaN. Therefore, we feel comfortable dropping all missing values from the data.

In [4]:
# Display all unique values in the 'home_ownership' column
unique_home_status = df['home_ownership'].unique()
unique_home_status

array(['MORTGAGE', 'RENT', 'OWN', 'ANY', 'OTHER', 'NONE', nan],
      dtype=object)

In [5]:
# Selecting categorical columns
categorical_cols = ['home_ownership']

# Perform One-Hot Encoding
# drop_first=True to avoid the "Dummy Variable Trap" 
df_cleaned = pd.get_dummies(df_cleaned, columns=categorical_cols, drop_first=True)

df_cleaned.head()

,delinq_2yrs,annual_inc,num_accts_ever_120_pd,pct_tl_nvr_dlq,dti,loan_amnt,emp_length,loan_status,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT
1,3.0,136000.0,1.0,90.0,20.63,30000.0,10+ years,Current,True,False,False,False,False
2,0.0,50000.0,0.0,100.0,29.62,21500.0,1 year,Fully Paid,False,False,False,False,True
4,0.0,88000.0,1.0,60.0,5.32,5000.0,10+ years,Current,True,False,False,False,False
5,0.0,38500.0,0.0,100.0,33.73,16150.0,10+ years,Charged Off,False,False,False,False,True
6,0.0,40000.0,0.0,100.0,19.11,18400.0,8 years,Current,False,False,False,False,True


We decided to encode home ownership status to have dummies for each status. While it would have been possible to assign a number to each status, we felt that it would hurt the interpretability of the variable, as there is no clear ranking for types of ownership. Many applicants would be unfairly hurt by a ranking of ownership, as renting versus owning have different trade-offs for individuals that may not be income related. Therefore, we feel it is unfair to rank the ownership, and multiple dummies would provide better interpretability and predictive power.

In [6]:
# Create the mapping dictionary based on the dataset categories
emp_map = {
    '< 1 year': 0, 
    '1 year': 1, 
    '2 years': 2, 
    '3 years': 3, 
    '4 years': 4,
    '5 years': 5, 
    '6 years': 6, 
    '7 years': 7, 
    '8 years': 8, 
    '9 years': 9, 
    '10+ years': 10
}

# Apply it to the 'emp_length' column
df_cleaned['emp_length'] = df_cleaned['emp_length'].map(emp_map)

# Check the result - it should now be 0.0 to 10.0
print(df_cleaned['emp_length'].value_counts())

emp_length
10    8127
2     2140
3     1889
0     1833
1     1525
4     1432
5     1403
6     1146
8     1042
7     1032
9      960
Name: count, dtype: int64


For employment length, we decided to map the data type to integers. We chose to assign '< 1 year' to 0, and '10+ years' to 10. We believe that this is the most logical ranking for employment length, as it keeps the integer values in the original dataset, while providing intuitive values for the non-integer values.

In [7]:
# Define which categories count as Default (1)
# Everything else will be Repaid (0)
default_categories = [
    'Charged Off', 
    'Default', 
    'Late (31-120 days)', 
    'Does not meet the credit policy. Status:Charged Off'
]

# Create the binary target column
df_cleaned['default'] = df_cleaned['loan_status'].apply(lambda x: 1 if x in default_categories else 0)
df_cleaned.drop('loan_status', axis=1, inplace=True)
df_cleaned.head()

,delinq_2yrs,annual_inc,num_accts_ever_120_pd,pct_tl_nvr_dlq,dti,loan_amnt,emp_length,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,default
1,3.0,136000.0,1.0,90.0,20.63,30000.0,10,True,False,False,False,False,0
2,0.0,50000.0,0.0,100.0,29.62,21500.0,1,False,False,False,False,True,0
4,0.0,88000.0,1.0,60.0,5.32,5000.0,10,True,False,False,False,False,0
5,0.0,38500.0,0.0,100.0,33.73,16150.0,10,False,False,False,False,True,1
6,0.0,40000.0,0.0,100.0,19.11,18400.0,8,False,False,False,False,True,0


In [8]:
# Report the final dimensions of the cleaned dataset
print(f"Final dimensions after cleaning and encoding: {df_cleaned.shape}")

Final dimensions after cleaning and encoding: (22529, 13)


The final dimensions of our dataset are $22,529 \times 13$. It should be noted we had to drop the 'ANY' home ownership column to avoid the dummy variable trap. We also dropped the 'loan_status' now that it has been encoded as a dummy variable.

### 1.4 Splitting Training and Test Data, and Standardization